In [2]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

Note: you may need to restart the kernel to use updated packages.


In [3]:
import warnings
warnings.simplefilter('ignore')

In [4]:
import os
import sys
import subprocess

In [5]:
def set_env(input_archive, temp_dir):

    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir, exist_ok=True)
        
        subprocess.run(['tar', '-xzf', input_archive, '-C', temp_dir], check=True)
    
    subprocess.run([
        sys.executable, 
        '-m', 
        'pip', 
        'install', 
        '--no-index', 
        '--find-links', 
        f'{temp_dir}/wheels', 
        'unsloth', 
        'trl', 
        'vllm', 
        'openai_harmony',
        'faiss-cpu', 
        'sentence-transformers'
    ], check=True)

In [6]:
set_env(
    input_archive='/kaggle/input/aimo-3-utils-cj/wheels.tar.gz', 
    temp_dir='/kaggle/tmp/setup'
)

Looking in links: /kaggle/tmp/setup/wheels
Processing /kaggle/tmp/setup/wheels/scikit_learn-1.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (from sentence-transformers)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scikit-plot 0.3.7 requires matplotlib>=1.4.0, which is not installed.
lime 0.2.0.1 requires matplotlib, which is not installed.
fastai 2.8.4 requires matplotlib, which is not installed.
yellowbrick 1.5 requires matplotlib!=3.0.0,>=2.0.2, which is not installed.
mlxtend 0.23.4 requires matplotlib>=3.0.0, which is not installed.
cuml-cu12 25.6.0 requires cuda-python<13.0a0,>=12.6.2, but you have cuda-python 13.1.1 which is incompatible.
fastai 2.8.4 requires fastcore<1.9,>=1.8.0, but you have fastcore 1.11.3 which is incompatible.
fastai 2.8.4 requires torch<2.9,>=1.10, but you have torch 2.9.0+cu128 which is incompatible.


In [7]:
subprocess.run(['ls', '/kaggle/tmp/setup/tiktoken_encodings'])

cl100k_base.tiktoken
o200k_base.tiktoken


CompletedProcess(args=['ls', '/kaggle/tmp/setup/tiktoken_encodings'], returncode=0)

In [8]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/kaggle/tmp/setup/tiktoken_encodings'

In [9]:
import gc
import re
import math
import time
import queue
import threading
import contextlib
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import pandas as pd
import polars as pl

from openai import OpenAI

from openai_harmony import (
    HarmonyEncodingName, 
    load_harmony_encoding, 
    SystemContent, 
    ReasoningEffort, 
    ToolNamespaceConfig, 
    Author, 
    Message, 
    Role, 
    TextContent, 
    Conversation
)

from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server

In [44]:
class CFG:
    difficulty_threshold_easy = 1.5  # Entropy threshold
    difficulty_threshold_hard = 3.5
    
    # ===== TIMING CONFIGURATION =====
    system_prompt = (
    'You are an elite mathematical problem solver with expertise at the International '
    'Mathematical Olympiad (IMO) level. Your goal is to find the correct answer through '
    'rigorous mathematical reasoning and tool calling.\n\n'
    
    '# Problem-Solving Approach:\n'
    '1. UNDERSTAND: Carefully read and rephrase the problem. Identify given data, '
    'required findings, and constraints. For Geometry, reframe problems as proofs or '
    'geometrical interpretations immediately.\n'
    '2. EXPLORE: Consider multiple strategies (theorems, patterns, analogous problems). '
    'For Combinatorics, define variables clearly, solve in terms of conditions, and '
    'apply integer constraints last to find the final answer.\n'
    '3. PLAN: Select the most promising approach. Incorporate various proof techniques '
    'such as Proof by Contradiction, Induction, or Pigeonhole Principle where applicable.\n'
    '4. EXECUTE: Work methodically. Handle Long Integer calculations with extreme care '
    'to avoid precision errors. Show all reasoning steps. **DO NOT hallucinate theorems, lemmas, or proofs—use the query_database tool ONLY when referencing mathematical results.** '
    'Apply retrieved theorems and their assumptions directly to solve the problem.\n'
    '5. VERIFY: BEFORE giving the final answer, verify exactly what the question asked '
    'versus what you are providing. Cross-check logic and arithmetic. \n\n'
    
    '# Mathematical Reasoning Principles:\n'
    '- BEWARE OF OVER-GENERALIZATION: Ensure every step is logically sound for the specific case.\n'
    '- Use Proof by Contradiction to test the validity of theorems and criteria.\n'
    '- Break complex problems into smaller, manageable sub-problems.\n'
    '- Look for patterns, symmetries, and special cases but do not assume they hold without proof.\n'
    '- Use concrete examples to build intuition before generalizing.\n'
    '- If stuck, try working backwards from the desired result.\n\n'
    
    '# Verification Requirements:\n'
    '- **Use query_database tool ONLY for verifying mathematical theorems, lemmas, and named results.** Do not use it for general problem solving or code execution.\n'
    '- When a theorem is retrieved, explicitly state its assumptions, constraints, and applicability conditions before using it in your solution.\n'
    '- Explicitly verify that the final numerical result answers the specific prompt question.\n'
    '- Cross-check arithmetic, especially for long integer calculations.\n'
    '- Test your answer with simple cases or special values when possible.\n'
    '- Ensure dimensional consistency and logical coherence.\n\n'
    
    '# Output Format:\n'
    'The final answer must be a non-negative integer between 0 and 99999.\n'
    'Place your final numerical answer inside \\boxed{}, e.g., \\boxed{42}\n\n'
    
    'Think step-by-step and show your complete reasoning process. Quality of reasoning '
    'is as important as the final answer.'
)
    
    tool_prompt = (
    "You are an IMO-level mathematician. You have access to a stateful environment with two specific tools: "
    "'python' and 'query_database'.\n\n"

    "### 1. Query Database TOOL\n"
    "Use <query_database(query string)> when you need to verify mathematical theory, lemmas, or identities. "
    "Before applying complex properties, use this tool to ensure you have the correct assumptions and constraints.\n"
    "- Use it for named lemmas (e.g., LTE Lemma, Burnside's Lemma, Chinese Remainder Theorem).\n"
    "- Use it for specific inequalities (e.g., Jensen's, Muirhead's, Cauchy-Schwarz).\n"
    "- Use it when you are unconfident about a theorem's statement or edge cases.\n\n"

    "### 2. PYTHON TOOL\n"
    "Use <python> to execute code in a stateful Jupyter notebook. Code persists between executions.\n"
    "- Perform complex arithmetic and long integer calculations that are error-prone manually.\n"
    "- Conduct brute-force verifications for small cases to find patterns.\n"
    "- Test conjectures or verify analytical results numerically.\n"
    "- Always use print() to display results. Write clear, well-commented code.\n\n"

    "### STRATEGY & GUIDELINES\n"
    "1. Explain your mathematical reasoning first. Code and theorems should support your logic, not replace it.\n"
    "2. If a problem involves deep theory, search the theorem library FIRST, then use the confirmed results to build your Python code.\n"
    "3. Be precise with variables. In Python, use descriptive names for specific mathematical use cases.\n"
    "4. When using retrieved theorems, pay close attention to their ASSUMPTIONS and CONSTRAINTS (e.g., 'only for positive integers')."
)
    
    preference_prompt = (
        'You have access to `math`, `numpy`, and `sympy` for:\n\n'
        
        '# Symbolic Computation (sympy):\n'
        '- Algebraic manipulation and simplification\n'
        '- Solving equations and systems of equations\n'
        '- Symbolic differentiation and integration\n'
        '- Number theory functions (primes, divisors, modular arithmetic)\n'
        '- Polynomial operations and factorization\n'
        '- Working with mathematical expressions symbolically\n\n'
        
        '# Numerical Computation (numpy):\n'
        '- Array operations and linear algebra\n'
        '- Efficient numerical calculations for large datasets\n'
        '- Matrix operations and eigenvalue problems\n'
        '- Statistical computations\n\n'
        
        '# Mathematical Functions (math):\n'
        '- Standard mathematical functions (trig, log, exp)\n'
        '- Constants like pi and e\n'
        '- Basic operations for single values\n\n'
        
        'Best Practices:\n'
        '- Use sympy for exact symbolic answers when possible\n'
        '- Use numpy for numerical verification and large-scale computation\n'
        '- Combine symbolic and numerical approaches: derive symbolically, verify numerically\n'
        '- Document your computational strategy clearly\n'
        '- Validate computational results against known cases or theoretical bounds'
    )
    
    served_model_name = 'gpt-oss'
    model_path = '/kaggle/input/gpt-oss-120b/transformers/default/1'
    
    kv_cache_dtype = 'fp8_e4m3'
    dtype = 'auto'

    high_problem_timeout = 900
    base_problem_timeout = 300

    notebook_limit = 17400
    server_timeout = 180

    session_timeout = 960
    jupyter_timeout = 6
    sandbox_timeout = 3

    stream_interval = 200
    context_tokens = 65536
    buffer_tokens = 512
    search_tokens = 32
    top_logprobs = 5
    batch_size = 256
    early_stop = 4
    attempts = 8
    workers = 16
    turns = 128
    seed = 42

    gpu_memory_utilization = 0.75
    temperature = 1.0
    min_p = 0.02
    eagle_model_path = "/kaggle/input/wenliang1990-gpt-oss-120b-eagle3-aimo3/transformers/default/3/gpt-oss-120b-eagle3-aimo3"

In [45]:
set_seed(CFG.seed)

In [46]:
import json
import faiss
import numpy as np
import re
from sentence_transformers import SentenceTransformer

class MathTheoremDB:
    def __init__(self, json_path: str, model_path: str):
        self.embedder = SentenceTransformer(model_path, device="cpu")
        self.search_count = 0
        
        print(f"Loading data from {json_path}...")
        with open(json_path, 'r') as f:
            full_data = json.load(f)
        
        dataset = full_data.get("dataset", {})
        
        # Merge all entities
        self.entries = []
        self.entries.extend(dataset.get("theorems", []))
        self.entries.extend(dataset.get("definitions", []))
        self.entries.extend(dataset.get("others", []))
        
        self.id_map = {item['id']: item for item in self.entries}
        
        # 1. Improved Indexing: Title Weighting
        # We index the title multiple times or emphasize it to ensure it drives retrieval
        processed_texts = []
        for item in self.entries:
            title = item.get("title", "Untitled")
            content_list = item.get("contents", item.get("content", []))
            content_str = " ".join(content_list) if isinstance(content_list, list) else str(content_list)
            
            # Boost the title by repeating it in the embedding string
            # This helps distinguish "Pigeonhole Principle" from a sentence containing "pigeon"
            processed_texts.append(f"Title: {title}. Content: {content_str[:400]}")

        print(f"Indexing {len(self.entries)} entries...")
        embeddings = self.embedder.encode(processed_texts, convert_to_numpy=True, show_progress_bar=True)
        faiss.normalize_L2(embeddings)
        self.index = faiss.IndexFlatL2(embeddings.shape[1])
        self.index.add(embeddings.astype('float32'))

    def _rerank_results(self, query: str, results: list) -> list:
        """Heuristic reranker to prioritize exact title matches."""
        query_words = set(re.findall(r'\w+', query.lower()))
        
        scored_results = []
        for res in results:
            score = 1.0  # Base similarity from FAISS
            title_lower = res['title'].lower()
            
            # Boost if query words are in title
            matches = sum(1 for word in query_words if word in title_lower)
            if matches > 0:
                score += (matches / len(query_words)) * 2.0
            
            # Penalty for very short titles that are just generic definitions (like "Bar")
            # unless it's a near-exact match
            if len(title_lower.split()) < 2 and matches < len(query_words):
                score -= 0.5
                
            scored_results.append((score, res))
            
        # Sort by new custom score
        scored_results.sort(key=lambda x: x[0], reverse=True)
        return [item[1] for item in scored_results]

    def search(self, query: str, top_k: int = 5):
        self.search_count += 1
        
        # Query Expansion: Help the model find Lemma/Theorem
        search_query = query
        if not any(keyword in query.lower() for keyword in ["theorem", "lemma", "formula", "inequality"]):
            search_query = f"{query} theorem lemma"

        query_vec = self.embedder.encode([search_query], convert_to_numpy=True).astype('float32')
        faiss.normalize_L2(query_vec)
        
        # Retrieve more than top_k for reranking
        distances, indices = self.index.search(query_vec, top_k * 3)
        
        initial_results = []
        for idx in indices[0]:
            if idx != -1 and idx < len(self.entries):
                item = self.entries[idx]
                content_data = item.get("contents", item.get("content", []))
                content_body = "\n".join(content_data) if isinstance(content_data, list) else str(content_data)
                
                initial_results.append({
                    "type": item.get("type", "statement"),
                    "title": item.get("title", "Unknown"),
                    "statement": content_body,
                    "ref_ids": item.get("ref_ids", [])
                })
        
        # Apply Reranking
        final_results = self._rerank_results(query, initial_results)
        
        # Attach related concepts only to the top results to save time/tokens
        for res in final_results[:top_k]:
            res['related_concepts'] = [self.id_map[rid].get("title") for rid in res.get("ref_ids", []) if rid in self.id_map][:5]
            
        return final_results[:top_k]
# import time
# import numpy as np

# # --- Initialization with your specific paths ---
# try:
#     print("Initializing Theorem DB...")
#     db = MathTheoremDB(
#         json_path='/kaggle/input/naturalproof/NaturalProof/naturalproofs_proofwiki.json',
#         model_path='/kaggle/input/all-minilm-l6-v2/transformers/default/1/all-MiniLM-L6-v2'
#     )
# except Exception as e:
#     print(f"Initialization Failed: {e}")
#     # Fallback for local testing if paths don't exist
#     db = None 



In [13]:
# import numpy as np
# from typing import List, Dict

# def fuzzy_match(expected: str, found: str) -> bool:
#     """Check if expected theorem name is significantly present in found title."""
#     expected = expected.lower().replace("-", " ").split()
#     found = found.lower().replace("-", " ")
#     # Check if all keywords of the expected theorem are in the found string
#     return all(word in found for word in expected)

# def evaluate_imo_theorems(db, categories: Dict[str, List[str]]):
#     all_ranks = []
#     category_scores = {}
    
#     print(f"{'Category':<15} | {'Query':<30} | {'Rank':<5} | {'Top Result Found'}")
#     print("-" * 85)
    
#     for cat_name, theorems in categories.items():
#         cat_ranks = []
#         for theorem_query in theorems:
#             # We assume the query is also the expected title for this test
#             results = db.search(theorem_query, top_k=10)
            
#             rank = 0
#             for i, res in enumerate(results, 1):
#                 if fuzzy_match(theorem_query, res['title']):
#                     rank = i
#                     break
            
#             reciprocal_rank = 1/rank if rank > 0 else 0
#             cat_ranks.append(reciprocal_rank)
#             all_ranks.append(reciprocal_rank)
            
#             rank_str = str(rank) if rank > 0 else "N/A"
#             top_title = results[0]['title'] if results else "None"
#             print(f"{cat_name:<15} | {theorem_query[:30]:<30} | {rank_str:<5} | {top_title[:30]}")
            
#         category_scores[cat_name] = np.mean(cat_ranks)

#     overall_mrr = np.mean(all_ranks)
#     hit_rate_1 = (sum(1 for r in all_ranks if r == 1.0) / len(all_ranks)) * 100

#     print("\n" + "="*30)
#     print(f"OVERALL HIT RATE @ 1: {hit_rate_1:.2f}%")
#     print(f"OVERALL MRR:          {overall_mrr:.4f}")
#     print("="*30)
    
#     print("\nPerformance by Category (MRR):")
#     for cat, score in category_scores.items():
#         print(f"{cat:<15}: {score:.4f}")

# # 2. Expanded IMO Dataset
# imo_dataset = {
#     "Algebra": ["AM-GM Inequality", "Cauchy-Schwarz Inequality", "Jensen's Inequality", "Vieta's Formulas"],
#     "Number Theory": ["Fermat's Little Theorem", "Chinese Remainder Theorem", "Lifting The Exponent Lemma", "Quadratic Reciprocity"],
#     "Geometry": ["Ptolemy's Theorem", "Ceva's Theorem", "Menelaus Theorem", "Power of a Point"],
#     "Combinatorics": ["Pigeonhole Principle", "Inclusion-Exclusion Principle", "Stars and Bars Theorem"]
# }

# # Run Evaluation
# evaluate_imo_theorems(db, imo_dataset)

In [47]:
class AIMO3Template:

    def __init__(self):

        pass

    def get_system_content(self, system_prompt: str, tool_config: ToolNamespaceConfig) -> SystemContent:

        return (
            SystemContent.new()
            .with_model_identity(system_prompt)
            .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH)
            .with_tools(tool_config)
        )

    def apply_chat_template(
        self, 
        system_prompt: str, 
        user_prompt: str, 
        tool_config: ToolNamespaceConfig
    ) -> list[Message]:

        system_content = self.get_system_content(system_prompt, tool_config)        
        system_message = Message.from_role_and_content(Role.SYSTEM, system_content)

        user_message = Message.from_role_and_content(Role.USER, user_prompt)

        return [system_message, user_message]

In [48]:
class AIMO3Sandbox:

    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:

        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count

            return ports

    def __init__(self, timeout: float):

        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None
        
        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'

        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]

        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True

        self.execute(
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def _format_error(self, traceback: list[str]) -> str:

        clean_lines = []

        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)

            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue

            clean_lines.append(clean_frame)

        return ''.join(clean_lines)

    def execute(self, code: str, timeout: float | None = None) -> str:

        client = self._client
        effective_timeout = timeout or self._default_timeout
        
        msg_id = client.execute(
            code, 
            store_history=True, 
            allow_stdin=False, 
            stop_on_error=False
        )

        stdout_parts = []
        stderr_parts = []
        
        start_time = time.time()

        while True:
            elapsed = time.time() - start_time

            if elapsed > effective_timeout:
                self._km.interrupt_kernel()

                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=1.0)

            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream':
                text = content.get('text', '')

                if content.get('name') == 'stdout':
                    stdout_parts.append(text)

                else:
                    stderr_parts.append(text)

            elif msg_type == 'error':
                traceback_list = content.get('traceback', [])

                stderr_parts.append(self._format_error(traceback_list))

            elif msg_type in {'execute_result', 'display_data'}:
                data = content.get('data', {})
                text = data.get('text/plain')

                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')

            elif msg_type == 'status':
                if content.get('execution_state') == 'idle':
                    break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)

        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr

        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    def close(self):

        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()

        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)

            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

    def reset(self):
        
        self.execute(
            '%reset -f\n'
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def __del__(self):

        self.close()

In [49]:
class AIMO3Tool:

    def __init__(self, local_jupyter_timeout: float, tool_prompt: str, sandbox=None , theorem_db=None):

        self._local_jupyter_timeout = local_jupyter_timeout
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox
        
        self._owns_session = sandbox is None
        
        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()
        
        self.theorem_db = theorem_db

    def _ensure_session(self):

        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)

    def _ensure_last_print(self, code: str) -> str:
        lines = code.rstrip().split('\n')
        if not lines: return code
        
        last_line = lines[-1]
        # If the last line is indented or already a print/import/comment, don't wrap it.
        if (last_line.startswith(' ') or last_line.startswith('\t') or 
            any(x in last_line for x in ['print', 'import', 'plt.', '#'])):
            return code
            
        lines[-1] = f'print({last_line.strip()})'
        return '\n'.join(lines)

    @property
    def instruction(self) -> str:

        return self._tool_prompt

    @property
    def tool_config(self) -> ToolNamespaceConfig:
        return ToolNamespaceConfig(
            name='math_tools', 
            description=self._tool_prompt, 
            tools=[
                {
                    'name': 'python', 
                    'description': 'Execute Python code in a stateful Jupyter environment for calculations and verification'
                },
                {
                    'name': 'query_database', 
                    'description': 'Search mathematical theorem database for named results (e.g., "Pigeonhole Principle", "Cauchy-Schwarz Inequality"). REQUIRED before citing any theorem or lemma.'
                }
            ]
        )

    def search_theorems(self, query: str):
        """Formats the vector search results into a clean reference card for the LLM."""
        if not self.theorem_db:
            return "Theorem database not initialized."

        results = self.theorem_db.search(query, top_k=2)
        
        if not results:
            return ("No specific theorem found. Relying on internal knowledge. "
                    "First correctly state the theorem along with its ASSUMPTIONS & CONSTRAINTS. "
                    "Then solve the question while taking care of cases and inequalities.")
        
        formatted = "### MATHEMATICAL REFERENCE CARDS\n\n"
        for i, res in enumerate(results, 1):
            # We use 'statement' and 'related_concepts' to match our new DB structure
            formatted += f"**[{i}] {res['type'].upper()}: {res['title']}**\n"
            formatted += f"**Formal Statement:**\n{res['statement']}\n"
            
            if res['related_concepts']:
                formatted += f"**Related Concepts:** {', '.join(res['related_concepts'])}\n"
            
            formatted += "---\n\n"
            
        return formatted

    def process_sync_plus(self, message: Message) -> list[Message]:
        recipient = message.recipient 
        raw_input = message.content[0].text

        # Robust routing for query_database
        is_query = any(keyword in recipient for keyword in ['query_database', 'search']) or \
                   (recipient == 'math_tools' and 'search' in raw_input.lower())

        if is_query:
            output = self.search_theorems(raw_input)
            return [self._make_response(output, name='query_database', channel=message.channel)]

        elif recipient in ['python', 'math_tools.python', 'math_tools']:
            self._ensure_session()
            final_script = self._ensure_last_print(raw_input)
            with self._execution_lock:
                try:
                    output = self._jupyter_session.execute(final_script)
                except Exception as exc:
                    output = f'[ERROR] Execution Failed: {exc}'
            return [self._make_response(output, name='python', channel=message.channel)]
        
        else:
            return [self._make_response(f"Unknown tool recipient: {recipient}", name='system')]

    def _make_response(self, output: str, name: str, channel: str | None = None) -> Message:
        content = TextContent(text=output)
        author = Author(role=Role.TOOL, name=name)
        message = Message(author=author, content=[content]).with_recipient('assistant')
        if channel: message = message.with_channel(channel)
        return message

    

In [50]:


class AIMO3Solver:

    def __init__(self, cfg, port: int = 8000):
    
        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'
        self.template = AIMO3Template()
        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()
    
        self._preload_model_weights()
        
        self.server_process = self._start_server()
    
        self.client = OpenAI(
            base_url=self.base_url, 
            api_key=self.api_key, 
            timeout=self.cfg.session_timeout
        )
    
        self._wait_for_server()
        self._initialize_kernels()
    
        self.notebook_start_time = time.time()
        self.problems_remaining = 50
        self.theorem_db = MathTheoremDB(
            json_path='/kaggle/input/naturalproof/NaturalProof/naturalproofs_proofwiki.json',
            model_path='/kaggle/input/all-minilm-l6-v2/transformers/default/1/all-MiniLM-L6-v2'
        )
          
    def _preload_model_weights(self) -> None:
    
        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start_time = time.time()
        
        files_to_load = []
        total_size = 0
    
        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)
    
                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)
    
        def _read_file(path: str) -> None:
    
            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass
    
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            list(executor.map(_read_file, files_to_load))
    
        elapsed = time.time() - start_time
        print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')
    
    def _start_server(self) -> subprocess.Popen:
        # Build the base command
        import json
        # Kill any existing vLLM processes first
        try:
            subprocess.run(['pkill', '-f', 'vllm.entrypoints.openai.api_server'], 
                          check=False, capture_output=True)
            time.sleep(2)  # Wait for cleanup
        except:
            pass
        spec_cfg = {
        "model": self.cfg.eagle_model_path,
        "num_speculative_tokens": 2,
        "draft_tensor_parallel_size": 1,
        "method": "eagle",
        }
        model_dir = os.path.abspath(self.cfg.model_path)
        cmd = [
            sys.executable, 
            '-m', 
            'vllm.entrypoints.openai.api_server', 
            '--seed', 
            str(self.cfg.seed), 
            # ... inside _start_server ...
            '--model', model_dir,
            '--served-model-name', 
            self.cfg.served_model_name, 
            '--tensor-parallel-size', 
            '1', 
            '--max-num-seqs', 
            str(self.cfg.batch_size), 
            '--gpu-memory-utilization', 
            str(self.cfg.gpu_memory_utilization), 
            '--host', 
            '0.0.0.0', 
            '--port', 
            str(self.port), 
            '--dtype', 
            self.cfg.dtype, 
            '--kv-cache-dtype', 
            self.cfg.kv_cache_dtype, 
            '--max-model-len', 
            str(self.cfg.context_tokens), 
            '--stream-interval', 
            str(self.cfg.stream_interval), 
            '--async-scheduling', 
            '--disable-log-stats', 
            '--enable-prefix-caching',
            '--speculative-config', json.dumps(spec_cfg),
        ]
    
        self.log_file = open('vllm_server.log', 'w')
    
        return subprocess.Popen(
            cmd, 
            stdout=self.log_file, 
            stderr=subprocess.STDOUT, 
            start_new_session=True
        )
    
    def _wait_for_server(self):
    
        print('Waiting for vLLM server...')
        start_time = time.time()
    
        for _ in range(self.cfg.server_timeout):
            return_code = self.server_process.poll()
    
            if return_code is not None:
                self.log_file.flush()
    
                with open('vllm_server.log', 'r') as log_file:
                    logs = log_file.read()
    
                raise RuntimeError(f'Server died with code {return_code}. Full logs:\n{logs}\n')
    
            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                print(f'Server is ready (took {elapsed:.2f} seconds).\n')
    
                return
    
            except Exception:
                time.sleep(1)
    
        raise RuntimeError('Server failed to start (timeout).\n')
    
    def _initialize_kernels(self) -> None:
    
        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start_time = time.time()
    
        self.sandbox_pool = queue.Queue()
    
        def _create_sandbox():
            
            return AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)
    
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [executor.submit(_create_sandbox) for _ in range(self.cfg.workers)]
    
            for future in as_completed(futures):
                self.sandbox_pool.put(future.result())
    
        elapsed = time.time() - start_time
        print(f'Kernels initialized in {elapsed:.2f} seconds.\n')
    
    def _scan_for_answer(self, text: str) -> int | None:
        """PHASE 1: Enhanced answer extraction with confidence tracking"""
        answers = []
        
        # Pattern 1: Boxed answer (highest priority - IMO standard)
        for match in re.finditer(r'\\boxed\s*\{\s*([0-9,]+)\s*\}', text):
            try:
                value = int(match.group(1).replace(',', ''))
                if 0 <= value <= 99999:
                    answers.append({
                        'value': value,
                        'pattern': 'boxed',
                        'position': match.start(),
                        'confidence': 0.95
                    })
            except ValueError:
                pass
        
        # Pattern 2: Explicit "final answer is X" (with flexible separators)
        for match in re.finditer(r'final\s+answer\s*(?:is|:|\=)\s*([0-9,]+)', text, re.IGNORECASE):
            try:
                value = int(match.group(1).replace(',', ''))
                if 0 <= value <= 99999:
                    answers.append({
                        'value': value,
                        'pattern': 'explicit',
                        'position': match.start(),
                        'confidence': 0.85
                    })
            except ValueError:
                pass
        
        if not answers:
            return None
        
        # Use LAST match by position (most recent in text = most likely final answer)
        answers.sort(key=lambda x: x['position'], reverse=True)
        return answers[0]['value']
    
    def _compute_mean_entropy(self, logprobs_buffer: list) -> float:
    
        if not logprobs_buffer:
            return float('inf')
    
        total_entropy = 0.0
        token_count = 0
    
        for top_logprobs_dict in logprobs_buffer:
            
            if not isinstance(top_logprobs_dict, dict):
                continue
            
            if not top_logprobs_dict:
                continue
            
            token_entropy = 0.0
            
            for token_str, log_prob in top_logprobs_dict.items():
                # Use clipping to prevent math domain errors or prob > 1.0
                prob = max(0.0, min(1.0, math.exp(log_prob)))
                
                if prob > 1e-9: # Ignore near-zero probabilities for stability
                    token_entropy -= prob * math.log2(prob)
            
            total_entropy += token_entropy
            token_count += 1
    
        if token_count == 0:
            return float('inf')
    
        return total_entropy / token_count
    
    def _is_execution_error(self, response: str) -> bool:
        """PHASE 1: Precise error classification vs. informational text"""
        if response.startswith('[ERROR]'):
            return True
        
        # Check for actual traceback format (line starting with File)
        if re.search(r'^\s*File\s+"', response, re.MULTILINE):
            return True
        
        # Check for exception types at line start (more specific than substring)
        if re.search(r'^(ValueError|TypeError|AttributeError|KeyError|IndexError|ZeroDivisionError|RuntimeError|SyntaxError|NameError|ImportError):', 
                     response, re.MULTILINE):
            return True
        
        return False
    
    def _process_attempt(
        self, 
        problem: str, 
        system_prompt: str, 
        attempt_index: int, 
        stop_event: threading.Event, 
        deadline: float
    ) -> dict:
    
        if stop_event.is_set() or time.time() > deadline:
            return {
                'Attempt': attempt_index + 1, 
                'Answer': None, 
                'Python Calls': 0, 
                'Python Errors': 0, 
                'Response Length': 0, 
                'Entropy': float('inf')
            }
    
        local_tool = None
        sandbox = None
        python_calls = 0
        theorem_calls = 0
        python_errors = 0
        total_tokens = 0
        final_answer = None
        
        logprobs_buffer = []
        
        # PHASE 1: Initialize attempt logging for debugging
        attempt_log = {
            'attempt_id': attempt_index,
            'start_time': time.time(),
            'errors': [],
            'tool_calls': 0,
            'extracted_answers': [],
            'final_answer': None
        }

        attempt_seed = int(self.cfg.seed * 1103515245 + 12345 + attempt_index * 2147483647) % (2**31)
    
        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)
    
            local_tool = AIMO3Tool(
                local_jupyter_timeout=self.cfg.jupyter_timeout, 
                tool_prompt=self.cfg.tool_prompt, 
                sandbox=sandbox,
                theorem_db=self.theorem_db
            )
    
            encoding = self.encoding
            messages = self.template.apply_chat_template(
                system_prompt, 
                problem, 
                local_tool.tool_config
            )
    
            conversation = Conversation.from_messages(messages)
    
            for _ in range(self.cfg.turns):
                if stop_event.is_set() or time.time() > deadline:
                    break
    
                prompt_ids = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
                max_tokens = self.cfg.context_tokens - len(prompt_ids)
    
                if max_tokens < self.cfg.buffer_tokens:
                    break
    
                stream = self.client.completions.create(
                    model=self.cfg.served_model_name, 
                    temperature=self.cfg.temperature, 
                    logprobs=self.cfg.top_logprobs, 
                    max_tokens=max_tokens, 
                    prompt=prompt_ids, 
                    seed=attempt_seed, 
                    stream=True, 
                    extra_body={
                        'min_p': self.cfg.min_p, 
                        'stop_token_ids': self.stop_token_ids, 
                        'return_token_ids': True
                    }
                )
    
                try:
                    token_buffer = []
                    text_chunks = []
                    # Inside _process_attempt, update the stream loop:
                    start_gen_time = None
                    tokens_in_this_turn = 0
                    
                    for chunk in stream:
                        if stop_event.is_set() or time.time() > deadline:
                            break
    
                        new_tokens = chunk.choices[0].token_ids
                        new_text = chunk.choices[0].text
    
                        if new_tokens:
                            if start_gen_time is None:
                                start_gen_time = time.time() # Mark start of first token
                            n = len(new_tokens)
                            tokens_in_this_turn += n  # ✅ Increment the turn counter
                            token_buffer.extend(new_tokens)
                            total_tokens += len(new_tokens)
                            text_chunks.append(new_text)
                            
                            chunk_logprobs = chunk.choices[0].logprobs
                            
                            if chunk_logprobs is not None:
                                if chunk_logprobs.top_logprobs:
                                    logprobs_buffer.extend(chunk_logprobs.top_logprobs)
    
                        if '}' in new_text:
                            search_text = ''.join(text_chunks[-self.cfg.search_tokens:])
                            answer = self._scan_for_answer(search_text)
    
                            if answer is not None:
                                final_answer = answer
                                attempt_log['extracted_answers'].append({
                                    'answer': answer,
                                    'source': 'stream_detection',
                                    'timestamp': time.time()
                                })
                                break
                    # Calculate TPS for this turn
                    if start_gen_time:
                        duration = time.time() - start_gen_time
                        if duration > 0:
                            tps = tokens_in_this_turn / duration
                            print(f"Turn TPS: {tps:.2f} tok/s | Tokens: {tokens_in_this_turn}")
    
                finally:
                    stream.close()
    
                if final_answer is not None:
                    break
    
                if not token_buffer:
                    break
    
                new_messages = encoding.parse_messages_from_completion_tokens(token_buffer, Role.ASSISTANT)
                conversation.messages.extend(new_messages)
                last_message = new_messages[-1]
    
                if last_message.channel == 'final':
                    answer_text = last_message.content[0].text
                    final_answer = self._scan_for_answer(answer_text)
                    if final_answer is not None:
                        attempt_log['extracted_answers'].append({
                            'answer': final_answer,
                            'source': 'final_channel',
                            'timestamp': time.time()
                        })
                    break
    
                # Inside the loop where you handle last_message.recipient:
                if last_message.recipient in ['python', 'query_database', 'math_tools.python', 'math_tools.query_database']:
                    if last_message.recipient == 'python':
                        python_calls += 1
                    elif 'query_database' in last_message.recipient:
                        theorem_calls += 1
                    tool_responses = local_tool.process_sync_plus(last_message)
                    response_text = tool_responses[0].content[0].text
    
                    if self._is_execution_error(response_text):
                        python_errors += 1
                        attempt_log['errors'].append(response_text[:150])
    
                    conversation.messages.extend(tool_responses)
    
        except Exception as exc:
# Add this improved error handling to your _process_attempt method
            error_msg = str(exc)
            error_type = type(exc).__name__
            print(f"❌ Attempt {attempt_index} failed: {error_type}: {error_msg}")
            
            # Log the full attempt context for debugging
            attempt_log['errors'].append({
                'type': error_type,
                'message': error_msg,
                'timestamp': time.time()
            })
            
            # Distinguish between different types of errors
            if 'timeout' in error_msg.lower() or 'timed out' in error_msg.lower():
                print("   → Request timeout - increase session_timeout")
            elif 'connection' in error_msg.lower():
                print("   → Connection issue - server might be overloaded")
            elif '35644' in error_msg:
                print("   → HTTP error 35644 - likely timeout or request format issue")
            elif 'openai' in error_msg.lower() or 'completions' in error_msg.lower():
                print("   → OpenAI client issue - check request parameters") 
            else:
                python_errors += 1  # Only count actual python execution errors
                print("   → Real python error - counted")
    
        finally:
            if sandbox is not None:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)
    
        mean_entropy = self._compute_mean_entropy(logprobs_buffer)
        
        # Finalize attempt log
        attempt_log['end_time'] = time.time()
        attempt_log['duration'] = attempt_log['end_time'] - attempt_log['start_time']
        attempt_log['final_answer'] = final_answer
    
        return {
            'Attempt': attempt_index + 1, 
            'Response Length': total_tokens, 
            'Python Calls': python_calls, 
            'Theorem Calls': theorem_calls, # Log this
            'Python Errors': python_errors, 
            'Entropy': mean_entropy, 
            'Answer': final_answer,
            'Log': attempt_log
        }
    
    def _select_answer(self, detailed_results: list) -> int:

        answer_weights = defaultdict(float)
        answer_votes = defaultdict(int)

        for result in detailed_results:
            answer = result['Answer']
            entropy = result['Entropy']
            
            if answer is not None:
                weight = 1.0 / max(entropy, 1e-9)
                
                answer_weights[answer] += weight
                answer_votes[answer] += 1

        scored_answers = []

        for answer, total_weight in answer_weights.items():
            scored_answers.append({
                'answer': answer, 
                'votes': answer_votes[answer], 
                'score': total_weight
            })

        scored_answers.sort(key=lambda x: x['score'], reverse=True)

        vote_data = []

        for item in scored_answers:
            vote_data.append((
                item['answer'], 
                item['votes'], 
                item['score']
            ))

        vote_dataframe = pd.DataFrame(
            vote_data, 
            columns=['Answer', 'Votes', 'Score']
        )

        vote_dataframe = vote_dataframe.round({'Score': 3})
        display(vote_dataframe)
        
        if not scored_answers:
            print('\nFinal Answer: 0\n')
            return 0

        final_answer = scored_answers[0]['answer']    
        print(f'\nFinal Answer: {final_answer}\n')

        return final_answer
    
    def solve_problem(self, problem: str) -> int:
    
        print(f'\nProblem: {problem}\n')
        
        problem_start_time = time.time()
        user_input = f'{problem} {self.cfg.preference_prompt}'
        
        # Initial budget (will be refined after first attempt entropy)
        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global
        problems_left_others = max(0, self.problems_remaining - 1)
        reserved_time = problems_left_others * self.cfg.base_problem_timeout
    
        initial_budget = time_left - reserved_time
        initial_budget = min(initial_budget, self.cfg.high_problem_timeout)
        initial_budget = max(initial_budget, self.cfg.base_problem_timeout)
    
        deadline = time.time() + initial_budget
    
        print(f'Initial Budget: {initial_budget:.2f} seconds | Deadline: {deadline:.2f}\n')
    
        tasks = []
    
        for attempt_index in range(self.cfg.attempts):
            tasks.append((self.cfg.system_prompt, attempt_index))
    
        detailed_results = []
        valid_answers = []
    
        stop_event = threading.Event()
    
        executor = ThreadPoolExecutor(max_workers=self.cfg.workers)
    
        try:
            futures = []
    
            for (system_prompt, attempt_index) in tasks:
                future = executor.submit(
                    self._process_attempt, 
                    user_input, 
                    system_prompt, 
                    attempt_index, 
                    stop_event, 
                    deadline
                )
    
                futures.append(future)
    
            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)
    
                    # 1. Track answers that meet the "Confidence Bar"
                    if result['Answer'] is not None:
                        # Only consider this for early stopping if entropy is low
                        if result['Entropy'] <= 1.0:
                            valid_answers.append(result['Answer'])
                        else:
                            print(f"Attempt {result['Attempt']} ignored for early stop (High Entropy: {result['Entropy']:.3f})")
        
                    # 2. Check consensus among high-confidence answers
                    counts = Counter(valid_answers).most_common(1)
        
                    if counts and counts[0][1] >= self.cfg.early_stop:
                        print(f"Early stop triggered! Consensus reached on: {counts[0][0]} with {counts[0][1]} high-confidence votes.")
                        stop_event.set()
                        for f in futures:
                            f.cancel()
                        break
    
                except Exception as exc:
                    print(f'Future failed: {exc}')
                    continue
    
        finally:
            stop_event.set()
            executor.shutdown(wait=True, cancel_futures=True)
            
            self.problems_remaining = max(0, self.problems_remaining - 1)
    
        if detailed_results:
            results_dataframe = pd.DataFrame(detailed_results)
            results_dataframe['Entropy'] = results_dataframe['Entropy'].round(3)
            results_dataframe['Answer'] = results_dataframe['Answer'].astype('Int64')
            
            display(results_dataframe)
    
        if not valid_answers:
            print('\nResult: 0\n')
    
            return 0
    
        return self._select_answer(detailed_results)
    
    def __del__(self):
    
        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()
    
        if hasattr(self, 'log_file'):
            self.log_file.close()
    
        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try:
                    sb = self.sandbox_pool.get_nowait()
                    sb.close()
    
                except Exception:
                    pass

In [31]:

    def _classify_difficulty(self, entropy: float) -> str:
        """Classify problem difficulty based on entropy"""
        if entropy < self.cfg.difficulty_threshold_easy:
            return 'easy'
        elif entropy < self.cfg.difficulty_threshold_hard:
            return 'medium'
        else:
            return 'hard'
    
    def _allocate_time_budget(self, entropy: float) -> float:
        """Allocate time budget based on problem difficulty"""
        difficulty = self._classify_difficulty(entropy)
        
        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global
        problems_left_others = max(0, self.problems_remaining - 1)
        reserved_time = problems_left_others * self.cfg.base_problem_timeout
        
        base_budget = time_left - reserved_time
        
        if difficulty == 'easy':
            allocated = min(base_budget * 0.3, self.cfg.base_problem_timeout)
        elif difficulty == 'medium':
            allocated = min(base_budget * 0.6, self.cfg.base_problem_timeout * 1.5)
        else:  # hard
            allocated = min(base_budget, self.cfg.high_problem_timeout)
        
        return max(allocated, self.cfg.base_problem_timeout)


In [51]:
solver = AIMO3Solver(CFG)


Loading model weights from /kaggle/input/gpt-oss-120b/transformers/default/1 into OS Page Cache...
Processed 26 files (65.28 GB) in 4.65 seconds.

Waiting for vLLM server...


RuntimeError: Server died with code 1. Full logs:
Skipping import of cpp extensions due to incompatible torch version 2.9.0+cu128 for torchao version 0.15.0+cu128             Please see https://github.com/pytorch/ao/issues/2919 for more info
INFO 02-07 05:31:09 [scheduler.py:216] Chunked prefill is enabled with max_num_batched_tokens=2048.
[1;36m(APIServer pid=9462)[0;0m INFO 02-07 05:31:09 [api_server.py:1977] vLLM API server version 0.11.2
[1;36m(APIServer pid=9462)[0;0m INFO 02-07 05:31:09 [utils.py:253] non-default args: {'host': '0.0.0.0', 'model': '/kaggle/input/gpt-oss-120b/transformers/default/1', 'seed': 42, 'max_model_len': 65536, 'served_model_name': ['gpt-oss'], 'gpu_memory_utilization': 0.75, 'kv_cache_dtype': 'fp8_e4m3', 'enable_prefix_caching': True, 'max_num_seqs': 256, 'async_scheduling': True, 'stream_interval': 200, 'speculative_config': {'model': '/kaggle/input/wenliang1990-gpt-oss-120b-eagle3-aimo3/transformers/default/3/gpt-oss-120b-eagle3-aimo3', 'num_specula

In [ ]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    global correct_count, total_count, predictions
    id_value = id_.item(0)
    question_text = question.item(0)
    
    gc.disable()
    
    final_answer = solver.solve_problem(question_text)
    # Store prediction
    predictions[id_value] = final_answer
    
    # Check accuracy if ground truth available
    total_count += 1
    if id_value in ground_truth:
        gt = ground_truth[id_value]
        is_correct = (final_answer == gt)
        if is_correct:
            correct_count += 1
        status = "✅" if is_correct else "❌"
        print(f"Answer: {final_answer} | Ground Truth: {gt} | {status}")
        print(f"📊 Running Accuracy: {correct_count}/{total_count} ({100*correct_count/total_count:.1f}%)")
    else:
        print(f"Answer: {answer}")
    
    print("------\n")
    
    
    gc.enable()
    gc.collect()
    
    return pl.DataFrame({'id': id_value, 'answer': final_answer})
    
df = pd.read_csv(
    "/kaggle/input/aimo-reference-problems-dataset/aimo reference problems/50 Problems/aimo50_Questions.csv" , encoding="utf-8",
    encoding_errors="ignore"
)

# Store ground truth answers for accuracy calculation (only in local mode)
ground_truth = dict(zip(df["id"], df["answer"])) if "answer" in df.columns else {}
df = df.iloc[:5]
# Create input file without answers
df.drop("answer", axis=1, errors="ignore").to_csv("reference.csv", index=False)
# Track predictions for accuracy calculation

predictions = {}
correct_count = 0
total_count = 0

In [ ]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
    
else:
    inference_server.run_local_gateway(
        ("reference.csv",)
    )
    # Print final accuracy summary
    if ground_truth and total_count > 0:
        print("\n" + "=" * 50)
        print("📊 FINAL ACCURACY SUMMARY")
        print("=" * 50)
        print(f"Correct: {correct_count}/{total_count}")
        print(f"Accuracy: {100*correct_count/total_count:.1f}%")
        print("=" * 50)
        
        # Show details
        print("\nDetails:")
        for qid, pred in predictions.items():
            if qid in ground_truth:
                gt = ground_truth[qid]
                status = "✅" if pred == gt else "❌"
                print(f"  {qid}: pred={pred}, gt={gt} {status}")


# Add this to your notebook after initialization fails:
with open('vllm_server.log', 'r') as f:
    print(f.read())  # Last 5000 chars